<a href="https://colab.research.google.com/github/TomazDrumond/Tom_Fly/blob/main/NB4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TomazDrumond/Tom_Fly/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
else:
    hf_token = os.environ.get("HF_TOKEN")

import pandas as pd

rel = "hf://datasets/FlyRank/internship-warehouse"

df_content = pd.read_parquet(f"{rel}/dim_content.parquet", storage_options={"token": hf_token})
df_clients = pd.read_parquet(f"{rel}/dim_clients.parquet", storage_options={"token": hf_token})
df_daily = pd.read_parquet(
    f"{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet",
    storage_options={"token": hf_token}
)

print("dim_content:", df_content.shape)
print("dim_clients:", df_clients.shape)
print("fact_content_daily_performance (month=2026-03):", df_daily.shape)


In [ ]:
import duckdb, pandas as pd

con = duckdb.connect()
con.register("daily", df_daily)
con.register("content", df_content)

# Build the same March-2026 page-month table from ML-04, filtered on real availability
page_month = con.sql("""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_month,
           SUM(gsc_clicks) AS clicks_month,
           AVG(gsc_avg_position) AS avg_position_month
    FROM daily
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

page_month = page_month.merge(
    df_content[["client_hash_id", "content_hash_id", "content_updated_date", "word_count"]],
    on=["client_hash_id", "content_hash_id"], how="left"
)
page_month["content_updated_date"] = pd.to_datetime(page_month["content_updated_date"])
snapshot_date = pd.Timestamp("2026-03-31")
page_month["days_since_update"] = (snapshot_date - page_month["content_updated_date"]).dt.days

visible = page_month[page_month["impressions_month"] > 0].copy()
visible["ctr_month"] = visible["clicks_month"] / visible["impressions_month"]

# --- Signal 1 (flag-linked: staleness behind the refresh flags) ---
visible["staleness_bucket"] = pd.cut(
    visible["days_since_update"],
    bins=[-1, 90, 180, 365, 99999],
    labels=["<90d", "90-180d", "180-365d", "365d+"]
)
signal1 = visible.groupby("staleness_bucket", observed=True).agg(
    n=("ctr_month", "size"),
    avg_ctr=("ctr_month", "mean")
)
print("Signal 1 — staleness vs CTR")
print(signal1)


In [ ]:
# --- Signal 2 (flag-linked: CTR-vs-position behind the CTR-fix logic) ---
visible["position_bucket"] = pd.cut(
    visible["avg_position_month"],
    bins=[0, 3, 10, 20, 999],
    labels=["1-3", "4-10", "11-20", "20+"]
)
signal2 = visible.groupby("position_bucket", observed=True).agg(
    n=("ctr_month", "size"),
    avg_ctr=("ctr_month", "mean")
)
print("Signal 2 — position vs CTR")
print(signal2)

**Signal 1 — staleness vs CTR: MIXED.** No pages in this slice are older than 365 days
(the panel's content is genuinely young), and within the buckets that do exist, CTR doesn't
decline cleanly with age — it spikes at 90-180 days (0.0201) before dropping back down at
180-365 days (0.0022), contradicting the simple "older content gets worse CTR" assumption
the refresh flag relies on. Given the small n in those two buckets (1,325 and 261), this
reads as a genuinely unclear signal rather than a real reversal, so staleness is kept out of
the score entirely — see the note below on why it was dropped, not just downweighted.

**Signal 2 — position vs CTR: CONFIRMED.** CTR falls steadily and substantially as position
worsens — 0.0106 (positions 1-3) down to 0.0019 (position 20+) — backed by large sample
sizes throughout (16k-82k rows per bucket). This is the strong, reliable signal, and the one
the rule leans on for its core logic.

**The rule, in plain words:** A page is worth reviewing if it's getting meaningful search
volume (100+ impressions this month) but converting far fewer of those impressions into
clicks than other pages ranking at the same position — meaning the problem is the page
itself, not its ranking.

**Reason code:** `underperforming_ctr`. **Action label:** `review_for_refresh`.

**Note on staleness:** `content_updated_date` in `dim_content` turned out to be a current
extract, not a point-in-time March snapshot — 84% of rows (148,782 / 176,738) showed an
"update" date *after* the March snapshot, producing impossible negative
`days_since_update` values. Combined with the MIXED signal verdict above, staleness was
dropped from the score entirely rather than kept as a soft weight, since it can't be
trusted for the large majority of rows in this slice.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import numpy as np, os

# Drop staleness from the score entirely — content_updated_date isn't trustworthy as of March
meaningful_volume = (visible["impressions_month"] >= 100).astype(int)

# Compute the CTR benchmark ONLY among meaningful-volume pages, using the mean (nonzero, per Signal 2)
benchmark_pool = visible[visible["impressions_month"] >= 100]
ctr_benchmark_by_bucket = benchmark_pool.groupby("position_bucket", observed=True)["ctr_month"].mean()

# .astype(float) is required — .map() on a Categorical column returns Categorical by default,
# which can't be compared against a float Series
visible["ctr_benchmark"] = visible["position_bucket"].map(ctr_benchmark_by_bucket).astype(float)
underperforming = (visible["ctr_month"] < visible["ctr_benchmark"]).astype(int)

visible["score"] = underperforming * meaningful_volume * visible["impressions_month"]
visible["reason_code"] = "underperforming_ctr"
visible["action"] = "review_for_refresh"

print("Rows with score > 0:", (visible["score"] > 0).sum(), "/", len(visible))
print("\nctr_benchmark by position_bucket:")
print(ctr_benchmark_by_bucket)

queue = visible.sort_values("score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
queue[["client_hash_id", "content_hash_id", "score", "reason_code", "action",
       "impressions_month", "avg_position_month", "ctr_month", "ctr_benchmark"]].to_csv(
    "work/outputs/baseline_action_score.csv", index=False
)
queue.head(10)

The queue writes 176,738 scored rows to `work/outputs/baseline_action_score.csv`, of which
68,272 (38.6%) score above zero — meaningful volume, underperforming CTR relative to their
position tier. The CTR benchmark is computed only from meaningful-volume pages (>=100
impressions) using the mean, not the median, since the full population's CTR is heavily
zero-inflated and would have produced a benchmark of 0 in several position buckets — silently
making every page "non-underperforming" by definition and forcing the score to 0 everywhere,
which is exactly what happened on the first attempt. The realized benchmarks (0.0036 for
positions 1-3, down to 0.0012 for 20+) mirror Signal 2's confirmed pattern almost exactly.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top20 = queue[queue["score"] > 0].head(20).reset_index(drop=True)
top20[["client_hash_id", "content_hash_id", "score", "reason_code", "action",
       "impressions_month", "avg_position_month", "ctr_month", "ctr_benchmark"]]

1. content_0e03de76 — review_for_refresh — why: 221k impressions, CTR 0.0033 vs benchmark
   0.0036, position 2.7 (near-top ranking, should convert far better) — wrong if: this is a
   comparison/definition page where users scan the SERP snippet and don't need to click.
2. content_44f34c0a — review_for_refresh — why: 212k impressions, CTR 0.00011 vs benchmark
   0.0032 — a near-total non-click page at position 7.3, the single largest gap on this list
   — wrong if: title/meta is badly mismatched to the query rather than the content itself,
   which would need a copy fix, not a content refresh.
3. content_8d7d99f1 — review_for_refresh — why: 203k impressions, CTR 0.0014 vs 0.0036 at
   position 2.6 — wrong if: seasonal/trending query where March happened to be off-peak
   intent for this specific page.
4. content_36e53e9c — review_for_refresh — why: barely underperforming (0.00124 vs 0.00125
   benchmark) at position 32.8, in the noisy 20+ bucket — wrong if: this is essentially
   noise, not a real signal — flagged as a weak pick in Section 4.
5. content_b99ea686 — review_for_refresh — why: 194k impressions, CTR 0.0019 vs 0.0032 at
   position 4.5 — wrong if: the page ranks for a broad/ambiguous keyword pulling impressions
   from unrelated intents.
6. content_4ffe1811 — review_for_refresh — why: CTR 0.0031 close to benchmark 0.0036 at
   position 2.3, high volume (187k) — wrong if: this gap is within normal noise for a
   near-benchmark page and shouldn't have been flagged at all.
7. content_acbcc847 — review_for_refresh — why: 171k impressions, CTR 0.0015 vs 0.0032,
   position 3.4 — wrong if: SERP features (featured snippet, PAA) are absorbing clicks
   above this result, which a content refresh wouldn't fix.
8. content_471d9cab — review_for_refresh — why: 165k impressions, CTR 0.0024 vs 0.0032,
   position 4.7 — wrong if: seasonal dip specific to March for this client's vertical.
9. content_fd2117c2 — review_for_refresh — why: 151k impressions, CTR 0.0027 vs 0.0032,
   position 3.4 — wrong if: gap this small reflects normal variance, not underperformance.
10. content_82e35c48 — review_for_refresh — why: CTR 0.00042 vs 0.00125 (20+ bucket) at
    position 22.6 — wrong if: position 20+ pages naturally have unstable, low-n benchmarks
    (check bucket size), making "underperforming" here less reliable than for top buckets.
11. content_34a70fea — review_for_refresh — why: 143k impressions, CTR 0.0003 vs 0.0032 at
    position 3.2 — a large, real gap — wrong if: this page recently changed URL/redirect and
    GSC data is still catching up to actual on-page performance.
12. content_e241d641 — review_for_refresh — why: CTR 0.0024 vs 0.0032, position 3.3 —
    wrong if: gap reflects a title-tag issue rather than the underlying content quality.
13. content_f43118e0 — review_for_refresh — why: CTR 0.0014 vs 0.0032, position 5.0 —
    wrong if: query intent skews informational-only, where low CTR is structurally normal.
14. content_f352b7cf — review_for_refresh — why: CTR 0.0021 vs 0.0032, position 3.3 —
    wrong if: gap this size falls within typical day-to-day CTR noise at this volume.
15. content_8e1334d6 — review_for_refresh — why: CTR ~0.0000074 (essentially zero clicks)
    on 135k impressions at position 4.5 — one of the clearest real candidates on the list —
    wrong if: tracking/tagging is broken for this specific page, not a content problem.
16. content_7c637314 — review_for_refresh — why: CTR 0.0006 vs 0.0032, position 5.8 —
    wrong if: this is a duplicate/near-duplicate of another page cannibalizing its own clicks.
17. content_df47d1b9 — review_for_refresh — why: barely underperforming (0.00124 vs
    0.00125) at position 24.4, in the noisy 20+ bucket — wrong if: this is noise, not signal
    — flagged as a weak pick in Section 4 alongside #4.
18. content_fe3fd342 — review_for_refresh — why: CTR 0.0030 close to benchmark 0.0032,
    position 3.1 — wrong if: gap this small is within normal variance.
19. content_fec55986 — review_for_refresh — why: CTR ~0.000008 (essentially zero clicks) on
    124k impressions at position 9.4 — another clear real candidate — wrong if: tracking is
    broken rather than the content itself failing.
20. content_545bb6cc — review_for_refresh — why: CTR 0.0023 vs 0.0036, position 2.6 —
    wrong if: gap reflects normal day-to-day variance at this volume rather than a real
    structural underperformance.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# Leakage check: confirm score inputs trace only to March 2026 + static dim_content fields
score_inputs = ["impressions_month", "avg_position_month", "ctr_month", "ctr_benchmark"]
print("Score built from:", score_inputs)
print("Source tables used: fact_content_daily_performance (month=2026-03, gsc_data_available IS TRUE), dim_content")
print("fact_content_query_90d used?", False, "(excluded — window overlaps sealed test month, per ML-04)")
print("Any future month (Apr-Jun 2026) used?", False)
print("Staleness / content_updated_date used in score?", False,
      "(dropped — 84% of rows had an update date after the March snapshot, an untrustworthy field for this window)")

# Check bucket sizes behind the two weak picks (#4 and #17), both in the noisy 20+ bucket
print("\n20+ bucket size (benchmark_pool):", (benchmark_pool["position_bucket"] == "20+").sum())

**Weak picks:** #4 (`content_36e53e9c`) and #17 (`content_df47d1b9`) both sit essentially
*at* their bucket's benchmark (0.00124 vs 0.00125), in the noisy 20+ position tier — this is
very likely benchmark noise rather than genuine underperformance, and both should be treated
as low-confidence flags rather than real refresh candidates.

**No future-window or label-derived inputs:** confirmed above — every feature in the score
(`impressions_month`, `avg_position_month`, `ctr_month`, `ctr_benchmark`) comes from March
2026 data only (`fact_content_daily_performance`, filtered on `gsc_data_available IS TRUE`).
`fact_content_query_90d` was excluded entirely, since its window overlaps the sealed test
month. `content_updated_date` was also excluded from the score, since it turned out to be a
current extract rather than a March-era snapshot — the discovery of that leak (a real bug,
not a hypothetical one) is documented in Section 1.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.